# Course 3 · Week 3 — Hands-on: Q-learning on Mars Rover

Reinforcement learning's first algorithm. The rover sits in a 1-D row of 6 cells. Two terminals: left end gives reward 100 (the big one), right end gives 40 (smaller, but closer for some starting states). The rover doesn't know any of this. It just gets rewards when it bumps into a terminal.

By the end you'll have:

1. Modeled a tiny 1-D environment with `step(s, a) → (s_next, r, done)`
2. Implemented epsilon-greedy action selection
3. Implemented the Q-learning update (the Bellman equation in code)
4. Trained for 2000 episodes and watched the Q-values converge to the optimal policy
5. Visualized Q-values per state
6. (Stretch) Varied the discount factor gamma and seen how it changes the policy

Estimated time: **45–60 minutes.** Q-learning is the cleanest RL algorithm — once it clicks, you'll see Bellman everywhere.


## Setup — the environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1-D Mars rover. 6 states in a row.
#  state:  0       1     2     3     4       5
#  reward: 100     0     0     0     0      40   (only at terminal states)
#  type:   TERM    --    --    --    --    TERM
N_STATES  = 6
N_ACTIONS = 2          # 0 = move left, 1 = move right
TERMINAL  = {0, 5}
REWARDS   = {0: 100.0, 5: 40.0}

def step(s, a):
    """Take action a from state s. Returns (s_next, reward, done)."""
    if s in TERMINAL:
        return s, 0.0, True
    s_next = max(0, s - 1) if a == 0 else min(N_STATES - 1, s + 1)
    return s_next, REWARDS.get(s_next, 0.0), s_next in TERMINAL

print(f"States: 0–{N_STATES-1} | Terminals at {sorted(TERMINAL)} with rewards {REWARDS}")
print(f"Quick test: step(2, 1) = {step(2, 1)}   expect (3, 0.0, False)")
print(f"Quick test: step(4, 1) = {step(4, 1)}   expect (5, 40.0, True)")


## Quick recap

**Q-value** Q(s, a) = the expected total reward from taking action `a` in state `s` and acting optimally afterward.

**Bellman equation:**

$$Q(s, a) = r(s') + \gamma \max_{a'} Q(s', a')$$

In words: the value of taking action `a` is the immediate reward plus a discounted look at the best you can do *next*. The discount factor `γ` (gamma) controls patience — `γ = 0` is "live for today", `γ = 1` is "no time preference". Most problems use 0.5 to 0.99.

**Q-learning update** (one TD step):

$$Q(s, a) \leftarrow Q(s, a) + \alpha \bigl[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \bigr]$$

It nudges the current Q-estimate toward the target (reward + discounted future value). With enough exploration and enough episodes, this converges to the true optimal Q.

**Epsilon-greedy:** with probability `epsilon` pick a random action (explore); otherwise pick `argmax Q(s, *)` (exploit). Without exploration the agent might get stuck in a sub-optimal habit. With *too much* exploration it never settles. Standard trick: start with high epsilon and decay over time.


## Exercise — Q-learning

In [ ]:
def epsilon_greedy(Q, s, epsilon):
    """Pick an action: random with probability epsilon, otherwise argmax."""
    # TODO: implement
    return 0


def q_learn(n_episodes=2000, alpha=0.1, gamma=0.5, epsilon=0.5, max_steps=100, seed=0):
    """Tabular Q-learning. Returns the final Q-table."""
    np.random.seed(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))
    for episode in range(n_episodes):
        s = np.random.choice([1, 2, 3, 4])  # random non-terminal start
        for _ in range(max_steps):
            a = epsilon_greedy(Q, s, epsilon)
            s_next, r, done = step(s, a)
            # Bellman update:
            #   target = r + (gamma * max(Q[s_next])  if not done else 0)
            #   Q[s, a] += alpha * (target - Q[s, a])
            # TODO: implement the update
            if done:
                break
            s = s_next
    return Q


Q = q_learn()
print("Final Q-table (rows = states 0..5, cols = [left, right]):")
print(np.round(Q, 2))

# Derive policy
policy = ["—" if s in TERMINAL else ("left" if Q[s, 0] >= Q[s, 1] else "right") for s in range(N_STATES)]
print(f"\nLearned policy: {policy}")
expected = ["—", "left", "left", "left", "right", "—"]
print(f"Expected:       {expected}")
assert policy == expected, "Policy should converge to: head left for states 1-3, right for state 4"
print("✓ Q-learning works")


## Visualize the Q-table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(np.arange(N_STATES) - 0.18, Q[:, 0], width=0.35, color="#4338ca", label="Q(s, left)")
ax.bar(np.arange(N_STATES) + 0.18, Q[:, 1], width=0.35, color="#10b981", label="Q(s, right)")
ax.set_xticks(range(N_STATES))
ax.set_xlabel("state"); ax.set_ylabel("Q-value")
ax.set_title("Q-values for each state-action pair")
ax.legend(); ax.grid(alpha=0.3)
plt.show()


## ⭐ Stretch — vary gamma

In [ ]:
# STRETCH: vary the discount factor gamma. What policy does the rover learn for each value?
# Hint: at high gamma the rover is patient (will go far for a bigger reward).
# At low gamma it's impatient (grabs the closer reward).

for gamma_test in [0.1, 0.3, 0.5, 0.9]:
    # TODO: rerun q_learn with this gamma, derive policy, print
    pass
